In [0]:
%pip install lxml curl_cffi

import json
from datetime import datetime, timezone
from lxml import html
from curl_cffi import requests
import random
import time


In [0]:

# ------------------------------------------------------------------
# 1. configs
# ------------------------------------------------------------------
S3_BUCKET_PATH = "s3a://aws-scraping-test-oks"

PRODUCT_URLS = [
    "https://varus.ua/pivo-4-5-0-355l-svitle-corona-extra-s-b",
    "https://varus.ua/pivo-0-44l-4-2-temne-pasterizovane-draught-guinness-zb",
    "https://varus.ua/pivo-svitle-lager-s11-mova-033l",
    "https://varus.ua/pivo-0-5l-4-6-svitle-estrella-damm-barselona-zb",
    "https://varus.ua/korm-dlya-kotiv-sheba-tunets-ta-ovochi-v-sousi-85-g",
    "https://varus.ua/forel-steyk-ohlazhdennaya-vesovaya",
    "https://varus.ua/krem-sir-philadelphia-original-61-195-g",
]

XPATH_PRODUCT = "//script[@type='application/ld+json' and contains(., 'Product')]/text()"
XPATH_BREADCRUMBS = "//script[@type='application/ld+json' and contains(., 'BreadcrumbList')]/text()"



In [0]:

# ------------------------------------------------------------------
# 2. scraping
# ------------------------------------------------------------------
def scrape_data(urls: list[str]) -> list[dict]:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }

    scraped_items = []

    for url in urls:
        try:
            response = requests.get(url, headers=headers, impersonate="chrome120", timeout=10)
            if response.status_code != 200:
                print(f"Пропущено {url} (Status: {response.status_code})")
                continue

            response.encoding = 'utf-8'
            tree = html.fromstring(response.text)

            categories = []
            breadcrumbs_scripts = tree.xpath(XPATH_BREADCRUMBS)

            for breadcrumbs_json in breadcrumbs_scripts:
                try:
                    breadcrumbs_data = json.loads(breadcrumbs_json)
                    item_list = breadcrumbs_data.get("itemListElement", [])
                    sorted_items = sorted(item_list, key=lambda x: x.get("position", 0))

                    for breadcrumb_item in sorted_items:
                        item_info = breadcrumb_item.get("item")
                        category_name = item_info.get("name") if isinstance(item_info, dict) else breadcrumb_item.get("name")
                        if category_name:
                            categories.append(category_name.strip())
                except json.JSONDecodeError:
                    continue

            category_path = " > ".join(categories) if categories else None

            scripts = tree.xpath(XPATH_PRODUCT)
            for raw_json in scripts:
                try:
                    product_data = json.loads(raw_json)
                    items = product_data if isinstance(product_data, list) else [product_data]

                    for item in items:
                        sku = item.get("sku")
                        name = item.get("name")

                        brand_obj = item.get("brand")
                        if isinstance(brand_obj, dict):
                            brand = brand_obj.get("name")
                        elif isinstance(brand_obj, str):
                            brand = brand_obj
                        else:
                            brand = None

                        offers_obj = item.get("offers")
                        price = None
                        if isinstance(offers_obj, dict):
                            price = offers_obj.get("price")
                        elif isinstance(offers_obj, list) and len(offers_obj) > 0:
                            price = offers_obj[0].get("price")

                        if sku:
                            scraped_items.append({
                                "sku": sku,
                                "name": name,
                                "brand": brand,
                                "categories": categories,
                                "category_path": category_path,
                                "price": price,
                                "url": url,
                                "scraped_at": datetime.now(timezone.utc).isoformat()
                            })
                except json.JSONDecodeError:
                    continue

            time.sleep(random.uniform(1.0, 2.0))

        except Exception as e:
            print(f"Помилка обробки {url}: {e}")
            continue

    print(f"Успішно зібрано {len(scraped_items)} записів з {len(urls)} сторінок.")
    return scraped_items


In [0]:
# scrape_data(PRODUCT_URLS)

In [0]:
# ------------------------------------------------------------------
# 3. S3 UPLOAD (via Unity Catalog External Location)
# ------------------------------------------------------------------
def upload_json_to_s3(data: list[dict], bucket_path: str):
    if not data:
        print("Немає даних для завантаження — пропускаємо запис у S3.")
        return

    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    timestamp = datetime.now(timezone.utc).strftime("%H%M%S")
    s3_path = f"{bucket_path}/raw/products/date={today}/data_{timestamp}.json"

    json_str = json.dumps(data, ensure_ascii=False, indent=2)

    try:
        dbutils.fs.put(s3_path, json_str, overwrite=True)
        print(f"Файл успішно завантажено: {s3_path}")
    except Exception as e:
        print(f"Помилка завантаження в S3: {e}")

In [0]:
# ------------------------------------------------------------------
# 4. MAIN LAUNCH
# ------------------------------------------------------------------
data = scrape_data(PRODUCT_URLS)
if data:
    upload_json_to_s3(data, S3_BUCKET_PATH)